In [1]:
import pandas as pd
import numpy as np
from datetime import date,timedelta
from dateutil.relativedelta import relativedelta
import random

In [2]:
import subprocess
subprocess.run(["pip", "install", "mysql-connector-python"], check=True)

CompletedProcess(args=['pip', 'install', 'mysql-connector-python'], returncode=0)

In [3]:
# DATE RANGE
start_date = date(2024,1,1)
end_date = date(2024,12,31)


# THE SHIFTS LIST
shifts = [
    {
        "shift_type": "Opening",
        "start": "04:00", "end": "05:00",
        "base_sales": {"Weekday": 250, "Friday": 350, "Weekend": 400}
    },
    
    {
        "shift_type": "Early Breakfast",
        "start": "05:00", "end": "07:00",
        "base_sales": {"Weekday": 1200, "Friday": 1400, "Weekend": 1500}
    },
    
    {
        "shift_type": "Late Breakfast",
        "start": "07:00", "end": "11:00",
        "base_sales": {"Weekday": 1300, "Friday": 1700, "Weekend": 2000}
    },
    
    {
        "shift_type": "Lunch",
        "start": "11:00", "end": "14:00",
        "base_sales": {"Weekday": 1300, "Friday": 2000, "Weekend": 2500}
    },
    
    {
        "shift_type": "Afternoon",
        "start": "14:00", "end": "17:00",
        "base_sales": {"Weekday": 1100, "Friday": 1800, "Weekend": 2500}
    },
    
    {
        "shift_type": "Dinner",
        "start": "17:00", "end": "20:00",
        "base_sales": {"Weekday": 1300, "Friday": 2200, "Weekend": 2500}
    },
    
    {
        "shift_type": "Late Night",
        "start": "20:00", "end": "00:00",
        "base_sales": {"Weekday": 1000, "Friday": 1300, "Weekend": 1500}
    },
    
    {
        "shift_type": "Overnight",
        "start": "00:00", "end": "04:00",
        "base_sales": {"Weekday": 800, "Friday": 1000, "Weekend": 1200}
    }
]

# THE MULTIPLIER DICTIONARIES

# The weather multiplier dictionary is a simple lookup dictionary 
# you give it a weather type, it gives back a number.

weather_multiplier = {'Sunny': 1.0, "Rainy": 0.95, "Snowstorm": 0.5}

# The outer key is the weather season. The inner dictionary holds the 
# probability of each weather type

# The golden rule - Every row adds up to 1.0

weather_chances = {
    
    "Spring": {"Sunny": 0.55, "Rainy": 0.43, "Snowstorm": 0.02},
    "Summer": {"Sunny": 0.72, "Rainy": 0.28, "Snowstorm": 0.00},
    "Fall" : {"Sunny": 0.55, "Rainy": 0.43, "Snowstorm": 0.02},
    "Winter": {"Sunny": 0.63, "Rainy": 0.08, "Snowstorm": 0.29}
    
}

# BUSINESS SEASONS

seasons = [
    
    {"name": "Summer Vacation", "start": (6,21), "end": (9,5), "multiplier": 1.3},
    {"name": "Christmas Season", "start": (12,15), "end": (12,24), "multiplier": 1.25},
    {"name": "Back to School", "start": (9,1), "end": (9,15), "multiplier": 0.85},
    {"name": "Winter Slump", "start": (1,1), "end": (2,28), "multiplier": 0.85},
]

# Here dates are tuples (6,21) means month 6 and date 21.
# we store dates as tuples here because we want to reuse them
# across any year

# PUBLIC HOLIDAYS
holidays = {
    (1,1): 0.60, # New years Day
    (7,1): 1.50, # Canada Day
    (12,25): 0.30, # Christmas day
    (12,26): 1.60, # Boxing day
    (12,31): 1.20, # New years Eve
}

# Here holidays is a dictionary where key is a tuple representing holiday date
# value: The sales multiplier for that day



In [4]:
# FUNTION 1 - get_weather_season()

def get_weather_season(d):
    
    month = d.month
    
    if month in [3,4,5]:
        
        return "Spring"
    
    elif month in [6,7,8]:
        
        return "Summer"
    
    elif month in [9,10,11]:
        
        return "Fall"
    
    else:
        return "Winter"

# What it does: You give it a date. It tells you which weather season 
# that date belongs to
get_weather_season(date(2024, 6, 1))

'Summer'

In [5]:
# FUNCTION 2 -  get_day_type()

def get_day_type(d):
    
    day_num = d.weekday()
    
    if day_num == 4:
        
        return "Friday"
    
    elif day_num >= 5:
        
        return "Weekend"
    
    else:
        
        return "Weekday"
    
get_day_type(date(2024,1,8))

'Weekday'

In [6]:
# FUNCTION 3 - get_season_multipler
# What this function does is that when we pass a date, we are actually
# checking whether that date falls into any of the business seasons

def get_season_multiplier(d):
    
    for season in seasons:
        
        s_start = date(d.year, season["start"][0], season["start"][1])
        s_end = date(d.year, season["end"][0], season["end"][1])
        
        if s_start <= d <= s_end:
            
            return season["multiplier"], season['name']
        
    return 1.0, "None"

get_season_multiplier(date(2024,1,1))
    
    

(0.85, 'Winter Slump')

In [7]:
# Generation Loop

all_rows = []
current_date = start_date

while current_date <= end_date:
    
    day_type = get_day_type(current_date)
    
    weather_season = get_weather_season(current_date)
    
    # pick today's weather
    chances = weather_chances[weather_season]
    weather = random.choices (
    
        population = list(chances.keys()),
        weights = list(chances.values())
    )[0]
    
    # holiday check
    month_day = (current_date.month,current_date.day)
    holiday_mult = holidays.get(month_day,1.0)
    is_holiday = month_day in holidays
    
    season_mult, season_name = get_season_multiplier(current_date)
    
    
    for shift in shifts:
        
        base = shift["base_sales"][day_type]
        wobble = np.random.uniform(0.90,1.10)
        w_mult = weather_multiplier[weather]
        
        actual_sales = round(base * w_mult * season_mult * holiday_mult * wobble,2)
        target_sales = round(base * w_mult * season_mult * holiday_mult, 2)
        sales_variance = round(actual_sales - target_sales, 2)
        
        all_rows.append({
            
            "date":             current_date,
            "day_of_week":      current_date.strftime("%A"),
            "day_type":         day_type,
            "month":            current_date.strftime("%B"),
            "quarter":          f"Q{(current_date.month - 1)//3+1}",
            "shift_type":       shift["shift_type"],
            "shift_start":      shift["start"],
            "shift_end":        shift["end"],
            "weather":          weather,
            "weather_season":   weather_season,
            "is_holiday":       is_holiday,
            "holiday_mult":     holiday_mult,
            "season_name":      season_name if season_name else "None",
            "season_mult":      season_mult,
            "base_sales":       base,
            "target_sales":     target_sales,
            "actual_sales":     actual_sales,
            "sales_variance":   sales_variance,
            
                
        })
    current_date += timedelta(days=1)

In [8]:
df = pd.DataFrame(all_rows)
df.to_csv("shift_sales.csv",index = False)
print(f"Done! {len(df)} rows saved.")

Done! 2928 rows saved.


In [9]:
print(df[df['season_mult'] == 1.0]['season_name'].value_counts())

None    1680
Name: season_name, dtype: int64


In [10]:
# LABOUR TABLE

labour_rows    = []
HOURLY_WAGE    = 17.49
SHIFT_HOURS    = 4
cost_per_staff = HOURLY_WAGE * SHIFT_HOURS

for _, row in df.iterrows():

    actual_sales = row["actual_sales"]
    day_type     = row["day_type"]
    is_holiday   = row["is_holiday"]

    if is_holiday:
        min_pct, max_pct = 0.17, 0.21
    elif day_type == "Weekend":
        min_pct, max_pct = 0.17, 0.23
    elif day_type == "Friday":
        min_pct, max_pct = 0.19, 0.23
    else:
        min_pct, max_pct = 0.21, 0.25

    actual_labour_pct  = round(np.random.uniform(min_pct, max_pct), 4)
    target_labour_cost = round(actual_sales * 0.23, 2)
    actual_labour_cost = round(actual_sales * actual_labour_pct, 2)
    labour_variance    = round(actual_labour_cost - target_labour_cost, 2)

    scheduled_staff = max(1, round(target_labour_cost / cost_per_staff))
    actual_staff    = max(1, round(actual_labour_cost  / cost_per_staff))
    labour_hours    = round(actual_staff * SHIFT_HOURS, 1)

    labour_rows.append({
        "date":               row["date"],
        "shift_type":         row["shift_type"],
        "day_type":           day_type,
        "actual_sales":       actual_sales,
        "target_labour_pct":  0.23,
        "actual_labour_pct":  actual_labour_pct,
        "target_labour_cost": target_labour_cost,
        "actual_labour_cost": actual_labour_cost,
        "labour_variance":    labour_variance,
        "scheduled_staff":    scheduled_staff,
        "actual_staff":       actual_staff,
        "labour_hours":       labour_hours,
        "is_holiday":         is_holiday,
    })

df_labour = pd.DataFrame(labour_rows)
df_labour.to_csv("labour.csv", index=False)
print(f"Done! {len(df_labour)} rows saved.")

Done! 2928 rows saved.


In [11]:
# Waste Table
import random

WASTE_REASONS = {
    
    "UHC Expiration":             0.40,
    "Overproduction":             0.25,
    "Expired Vegetables":         0.20,
    "Dropped Food":               0.08,
    "Wrong Order":                0.05,
    "Delivery Driver No Show:":   0.02,
    
}

# Waste % range per shift type
# Busy shifts = higher waste, slow shifts = lower waste
WASTE_PCT_RANGE = {
    "Opening":         (0.01, 0.02),
    "Early Breakfast": (0.02, 0.04),
    "Late Breakfast":  (0.03, 0.05),
    "Lunch":           (0.04, 0.06),
    "Afternoon":       (0.01, 0.03),
    "Dinner":          (0.04, 0.07),
    "Late Night":      (0.02, 0.04),
    "Overnight":       (0.01, 0.02),
}

TARGET_WASTE_PCT = 0.03   # 3% target

waste_rows = []

for _, row in df.iterrows():

    actual_sales = row["actual_sales"]
    shift_type   = row["shift_type"]

    # Get waste % range for this shift
    min_pct, max_pct = WASTE_PCT_RANGE[shift_type]

    # Pick a random waste percentage
    actual_waste_pct = round(np.random.uniform(min_pct, max_pct), 4)

    # Calculate waste costs
    target_waste_cost = round(actual_sales * TARGET_WASTE_PCT, 2)
    actual_waste_cost = round(actual_sales * actual_waste_pct, 2)
    waste_variance    = round(actual_waste_cost - target_waste_cost, 2)

    # Pick a waste reason based on probabilities
    waste_reason = random.choices(
        population = list(WASTE_REASONS.keys()),
        weights    = list(WASTE_REASONS.values())
    )[0]

    # Is waste over the 5% acceptable limit?
    is_over_target = actual_waste_pct > 0.05

    waste_rows.append({
        "date":               row["date"],
        "shift_type":         shift_type,
        "day_type":           row["day_type"],
        "actual_sales":       actual_sales,
        "target_waste_pct":   TARGET_WASTE_PCT,
        "actual_waste_pct":   actual_waste_pct,
        "target_waste_cost":  target_waste_cost,
        "actual_waste_cost":  actual_waste_cost,
        "waste_variance":     waste_variance,
        "waste_reason":       waste_reason,
        "is_over_target":     is_over_target,
    })

df_waste = pd.DataFrame(waste_rows)
df_waste.to_csv("waste.csv", index=False)
print(f"Done! {len(df_waste)} rows saved.")

Done! 2928 rows saved.


In [13]:
import pandas as pd

# Read and resave each CSV with correct formatting for MySQL
df_sales  = pd.read_csv("shift_sales.csv")
df_labour = pd.read_csv("labour.csv")
df_waste  = pd.read_csv("waste.csv")

# Save with windows line endings
df_sales.to_csv("shifts_sales_clean.csv",  index=False, lineterminator='\r\n')
df_labour.to_csv("labour_clean.csv",       index=False, lineterminator='\r\n')
df_waste.to_csv("waste_clean.csv",         index=False, lineterminator='\r\n')

print("Done — 3 clean CSV files saved")

Done — 3 clean CSV files saved
